# Day 08 - 特征工程：让数据说话在机器学习中，有一句名言：**"数据和特征决定了机器学习的上限，而模型和算法只是逼近这个上限。"**今天我们来学习 **特征工程 (Feature Engineering)** —— 如何从原始数据中提取出更有用的信息。---## 今天的目标1. 理解什么是特征工程2. 学习日期特征提取3. 掌握特征缩放方法4. 通过实验验证特征工程的效果

## 1. 什么是特征工程？**特征工程** 就是把原始数据转换成对模型更有用的格式。比如：原始数据只有日期 "2024-07-15"，我们可以从中提取：- 月份：7（夏天）- 季节：夏季- 是否周末：否- 年中第几天：197这些新特征可能帮助模型更好地理解数据！

In [ ]:
# 导入库import pandas as pdimport numpy as npimport matplotlib.pyplot as pltfrom sklearn.linear_model import LogisticRegressionfrom sklearn.model_selection import cross_val_scorefrom sklearn.preprocessing import StandardScaler, MinMaxScalerplt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei"]plt.rcParams["axes.unicode_minus"] = False# 读取数据df = pd.read_csv("data/weather_sample.csv", encoding="utf-8-sig")df["日期"] = pd.to_datetime(df["日期"])print("原始特征:", list(df.columns))df.head()

## 2. 日期特征提取从日期中提取时间信息是非常常见的特征工程。

In [ ]:
# 从日期中提取新特征df["月份"] = df["日期"].dt.monthdf["日"] = df["日期"].dt.daydf["星期"] = df["日期"].dt.dayofweek  # 0=周一, 6=周日df["是否周末"] = (df["星期"] >= 5).astype(int)df["年中第几天"] = df["日期"].dt.dayofyear# 季节特征（用月份判断）def get_season(month):    if month in [3, 4, 5]:        return 0  # 春    elif month in [6, 7, 8]:        return 1  # 夏    elif month in [9, 10, 11]:        return 2  # 秋    else:        return 3  # 冬df["季节"] = df["月份"].apply(get_season)print("新增特征后的数据:")df.head()

In [ ]:
# 可视化：不同月份的下雨概率monthly_rain = df.groupby("月份")["是否下雨"].mean()plt.figure(figsize=(10, 5))plt.bar(monthly_rain.index, monthly_rain.values, color="steelblue")plt.xlabel("月份", fontsize=14)plt.ylabel("下雨概率", fontsize=14)plt.title("各月份下雨概率", fontsize=16)plt.xticks(range(1, 13))plt.grid(True, alpha=0.3, axis="y")plt.tight_layout()plt.show()

## 3. 特征缩放 (Feature Scaling)不同特征的数值范围差别很大：- 温度：-10 ~ 40- 湿度：20 ~ 99- 气压：990 ~ 1030- 月份：1 ~ 12有些算法（如 K近邻、SVM）对数值范围很敏感，需要 **缩放**。**标准化 (StandardScaler)**：均值=0，标准差=1**归一化 (MinMaxScaler)**：缩放到 0~1

In [ ]:
# 对比两种缩放方法features_to_scale = ["温度", "湿度", "风速", "气压"]X_raw = df[features_to_scale].values# 标准化scaler_std = StandardScaler()X_std = scaler_std.fit_transform(X_raw)# 归一化scaler_minmax = MinMaxScaler()X_minmax = scaler_minmax.fit_transform(X_raw)fig, axes = plt.subplots(1, 3, figsize=(15, 4))axes[0].boxplot(X_raw, labels=features_to_scale)axes[0].set_title("原始数据")axes[0].grid(True, alpha=0.3)axes[1].boxplot(X_std, labels=features_to_scale)axes[1].set_title("标准化 (StandardScaler)")axes[1].grid(True, alpha=0.3)axes[2].boxplot(X_minmax, labels=features_to_scale)axes[2].set_title("归一化 (MinMaxScaler)")axes[2].grid(True, alpha=0.3)plt.tight_layout()plt.show()

## 4. 实验：特征工程真的有用吗？让我们做一个对比实验：- **实验A**：只用原始特征（湿度、风速、气压）- **实验B**：加上日期特征（月份、是否周末、季节）- **实验C**：加上日期特征 + 标准化

In [ ]:
# 实验A：原始特征features_A = ["湿度", "风速", "气压"]X_A = df[features_A]y = df["是否下雨"]model = LogisticRegression(random_state=42)score_A = cross_val_score(model, X_A, y, cv=5, scoring="accuracy").mean()print(f"实验A (原始特征):     准确率 = {score_A*100:.1f}%")# 实验B：加上日期特征features_B = ["湿度", "风速", "气压", "月份", "是否周末", "季节"]X_B = df[features_B]score_B = cross_val_score(model, X_B, y, cv=5, scoring="accuracy").mean()print(f"实验B (+日期特征):     准确率 = {score_B*100:.1f}%")# 实验C：加上日期特征 + 标准化scaler = StandardScaler()X_C = scaler.fit_transform(X_B)score_C = cross_val_score(model, X_C, y, cv=5, scoring="accuracy").mean()print(f"实验C (+标准化):       准确率 = {score_C*100:.1f}%")print(f"\n结论: 特征工程让准确率从 {score_A*100:.1f}% 提升到了 {score_C*100:.1f}%！")

In [ ]:
# 可视化对比experiments = ["原始特征", "+日期特征", "+标准化"]scores = [score_A, score_B, score_C]plt.figure(figsize=(8, 5))bars = plt.bar(experiments, [s*100 for s in scores],              color=["lightcoral", "gold", "mediumseagreen"])for bar, score in zip(bars, scores):    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,             f"{score*100:.1f}%", ha="center", fontsize=14)plt.ylabel("准确率 (%)", fontsize=14)plt.title("特征工程效果对比", fontsize=16)plt.ylim(0, 100)plt.tight_layout()plt.show()

---## 你学到了什么？| 概念 | 说明 ||------|------|| 特征工程 | 从原始数据中提取更有用的信息 || 日期特征 | 从时间中提取月份、季节等 || 标准化 | 均值=0，标准差=1 || 归一化 | 缩放到 0~1 || 对比实验 | 验证特征工程是否有效 |---## 挑战任务（选做）1. 构造一个新特征：温差（当日最高温 - 最低温），看看是否有帮助2. 试试把连续的温度分成几个区间（如寒冷/凉爽/温暖/炎热），作为新特征3. 研究一下：什么时候需要标准化，什么时候不需要？